# Vicuña · T09 y arquitectura

Parte de la mejor línea de T08: `Net` del starter, pérdida `rmse`, HR recortada (2 semillas a 25 épocas: score_oficial 0.44095).

Selección hacia adelante, **un cambio a la vez** sobre la mejor configuración aceptada:

1. GFS interpolado a la grilla WRF como entrada del decodificador (skip).
2. Predecir residuo sobre GFS (T09): residuo de anomalías y residuo crudo.
3. Pendiente, orientación y relieve relativo del DEM.
4. Decodificador residual más profundo.
5. Climatología con más peso en mayo y junio.
6. Reintento del skip sobre la configuración vigente.

Cada candidato: semillas 0 y 1, 25 y 60 épocas (época final fija), validación temporal 16–30 jun. Se acepta si, con la misma semilla, **las dos** mejoran y la media sube al menos 0.001. La selección usa la validación, así que el score final es algo optimista.

Al final: ensamble de 5 semillas en validación, reentrenamiento con todos los datos y CSV. No se envía desde aquí.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def find(name):
    # Busca en Kaggle, en la carpeta data/ del repo (corriendo desde train/ o desde la raíz) y en la carpeta actual
    hits = (glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'../data/**/{name}', recursive=True)
            + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True))
    assert hits, f'No encuentro {name}'
    return hits[0]

tr = np.load(find('train.npz'))
te = np.load(find('test.npz'))
st = np.load(find('static.npz'))

Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
VARS = ['RH', 'T', 'U', 'V']; UNITS = ['%', 'K', 'm/s', 'm/s']

print('Train  X (GFS):', Xtr.shape, ' Y (WRF):', Ytr.shape, ' horas:', ttr[0], '→', ttr[-1])
print('Test   X (GFS):', Xte.shape, ' horas:', tte[0], '→', tte[-1])
print('DEM:', dem.shape, f'{dem.min():.0f}–{dem.max():.0f} m')

In [ ]:
# COPIA EXACTA de train/validacion.py: no editar aquí
"""
Validación temporal con la métrica del concurso (T02 · HU-09). Solo NumPy.

Corte temporal fijo: validación = últimas dos semanas de junio (desde 2024-06-16 00 UTC);
entrenamiento = todo lo anterior. Nunca se hace split aleatorio.

Hay dos scores y no son comparables entre sí:
  score_local    el del starter: cada RMSE se divide entre el RMSE de la climatología de train
                 medido en la propia validación (climatología = 0). Todas las horas.
  score_oficial  fórmula del leaderboard, con sus constantes fijas, sobre las horas que Kaggle
                 evalúa (00, 03, ..., 21 UTC). Es el que se usa para comparar experimentos.

El notebook lleva una copia exacta de este archivo en una celda para correr solo en Kaggle;
tests/test_validacion.py comprueba que ambas sean iguales.
"""
import numpy as np

VARS = ("RH", "T", "U", "V")
INICIO_VAL = "2024061600"
ESCALAS_OFICIALES = np.array([25.360, 2.0652, 2.6776, 2.0876])
HORAS_EVALUADAS = tuple(range(0, 24, 3))


def horas(times):
    """Hora UTC (0-23) de cada marca 'YYYYMMDDHH'."""
    return np.array([int(str(t)[-2:]) for t in times])


def corte_temporal(times, inicio_val=INICIO_VAL):
    """Máscaras booleanas (train, val): val son las marcas desde `inicio_val` en adelante."""
    val = np.asarray(times).astype(str) >= inicio_val
    trn = ~val
    if not trn.any() or not val.any():
        raise ValueError(f"El corte {inicio_val} deja vacío el entrenamiento o la validación.")
    return trn, val


def climatologia(Y, times):
    """Media por hora UTC: (24, filas, columnas, variables). Exige datos en las 24 horas."""
    h = horas(times)
    faltan = sorted(set(range(24)) - set(h.tolist()))
    if faltan:
        raise ValueError(f"No hay datos para las horas UTC {faltan}.")
    return np.stack([Y[h == k].mean(0) for k in range(24)])


def rmse_por_variable(pred, real):
    """RMSE de cada variable sobre todas las horas y celdas. Entradas (N, filas, columnas, 4)."""
    if pred.shape != real.shape:
        raise ValueError(f"Formas distintas: pred {pred.shape} y real {real.shape}.")
    rmse = np.sqrt(((pred - real) ** 2).mean(axis=(0, 1, 2)))
    if not np.isfinite(rmse).all():
        raise ValueError("Hay NaN o infinitos en la predicción o en el valor real.")
    return rmse


def score(rmse, escalas):
    """1 - media de rmse/escala; las cuatro variables pesan igual."""
    return float(1 - (rmse / escalas).mean())


def escala_climatologia(Y, times, inicio_val=INICIO_VAL):
    """RMSE en validación de la climatología calculada solo con train (denominador de score_local)."""
    t = np.asarray(times).astype(str)
    trn, val = corte_temporal(t, inicio_val)
    clim = climatologia(Y[trn], t[trn])
    return rmse_por_variable(clim[horas(t[val])], Y[val])


def validar(pred_val, Y, times, inicio_val=INICIO_VAL):
    """
    Evalúa una predicción de la validación.

    pred_val : (N_val, 70, 35, 4) en unidades físicas, en el orden de las horas de validación.
    Y, times : target y marcas de TODO train.npz (de ahí salen el corte y la climatología).
    """
    t = np.asarray(times).astype(str)
    _, val = corte_temporal(t, inicio_val)
    real, t_val = Y[val], t[val]
    rmse = rmse_por_variable(pred_val, real)
    evaluadas = np.isin(horas(t_val), HORAS_EVALUADAS)
    rmse_3h = rmse_por_variable(pred_val[evaluadas], real[evaluadas])
    return {
        "score_local": score(rmse, escala_climatologia(Y, t, inicio_val)),
        "rmse": {v: float(r) for v, r in zip(VARS, rmse)},
        "score_oficial": score(rmse_3h, ESCALAS_OFICIALES),
        "rmse_horas_evaluadas": {v: float(r) for v, r in zip(VARS, rmse_3h)},
        "horas_val": int(val.sum()),
        "horas_evaluadas": int(evaluadas.sum()),
        "primera": str(min(t_val)),
        "ultima": str(max(t_val)),
    }

In [ ]:
def windows(X, t):
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        prev = i
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out   # (n, 3, 17, 13, 4)

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)
print(Wtr.shape, Wte.shape)

In [ ]:
import json, platform
assert torch.cuda.is_available(), 'Este notebook necesita GPU'
dev = 'cuda'
QUICK = False                      # True: prueba corta de que todo corre
trn, val = corte_temporal(ttr)
Wtr, Wte = (Wtr.astype(np.float32), Wte.astype(np.float32))
print(torch.cuda.get_device_name(0), '·', platform.python_version(), np.__version__, torch.__version__)
print(f'train {trn.sum()} h · val {val.sum()} h')
def to_t(a): return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)
ESC = to_t(ESCALAS_OFICIALES)

In [ ]:
# Grilla WRF dentro de la grilla GFS (como en el starter): GFS fila 0 = norte, WRF y DEM fila 0 = sur
glat, glon = st['gfs_lats'], st['gfs_lons']
wlat, wlon = st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = to_t(np.stack([gx, gy], -1)[None])                      # (1,70,35,2)
DEM = to_t((dem - dem.mean()) / dem.std())[None, None]

def media_local(a, k):
    """Media en una ventana k x k con borde reflejado."""
    p = np.pad(a, k // 2, mode='reflect')
    return np.lib.stride_tricks.sliding_window_view(p, (k, k)).mean((-1, -2))

def relieve(dem):
    """Pendiente (norte, este, magnitud), orientación (sin, cos) y relieve relativo (~15 y ~45 km)."""
    z = dem.astype(np.float64)
    dn, de = np.gradient(z)                                    # fila crece hacia el norte, columna al este
    mag = np.hypot(dn, de)
    unit = np.where(mag > 0, mag, 1.0)
    capas = [dn, de, mag, de / unit, dn / unit, z - media_local(z, 5), z - media_local(z, 15)]
    return np.stack([(c - c.mean()) / c.std() for c in capas])

TERRENO = to_t(relieve(dem))[None]                             # (1,7,70,35)

def interpolar(a):
    """(B,C,17,13) GFS -> (B,C,70,35) WRF, bilineal."""
    return F.grid_sample(a, GRID.expand(len(a), -1, -1, -1), align_corners=True)

class Bloque(nn.Module):
    def __init__(self, w, d):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(w, w, 3, padding=d, dilation=d), nn.GELU(),
                               nn.Conv2d(w, w, 3, padding=d, dilation=d))
    def forward(self, x):
        return F.gelu(x + self.f(x))

class Net(nn.Module):
    """Con skip=terreno=profundo=False es exactamente la red del starter."""
    def __init__(self, skip=False, terreno=False, profundo=False, w=64):
        super().__init__()
        self.skip, self.terreno = skip, terreno
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        c = w + 9 + (12 if skip else 0) + (7 if terreno else 0)
        if profundo:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     *[Bloque(w, d) for d in (1, 2, 4, 1)], nn.Conv2d(w, 4, 1))
        else:
            self.dec = nn.Sequential(nn.Conv2d(c, w, 3, padding=1), nn.GELU(),
                                     nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):                     # x: (B,3,17,13,4), h: (B,) hora UTC
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        partes = [interpolar(self.enc(torch.cat([x, hh], 1))), DEM.expand(B, -1, -1, -1),
                  self.static.expand(B, -1, -1, -1)]
        if self.skip:
            partes.append(interpolar(x))
        if self.terreno:
            partes.append(TERRENO.expand(B, -1, -1, -1))
        return self.dec(torch.cat(partes, 1)).permute(0, 2, 3, 1)  # (B,70,35,4) anomalía normalizada

In [ ]:
BASE = dict(skip=False, residuo=None, terreno=False, profundo=False, peso_mj=1)

def clim_ponderada(A, t, peso):
    """Media por hora UTC en la que las horas de mayo y junio pesan `peso` (1 = climatología de T02)."""
    if peso == 1:
        return climatologia(A, t)
    h = horas(t)
    w = np.where(np.array([s[4:6] in ('05', '06') for s in t]), float(peso), 1.0)
    return np.stack([np.average(A[h == k], axis=0, weights=w[h == k]) for k in range(24)]).astype(np.float32)

def datos_de(mascara, cfg):
    """Normalización y climatologías SOLO con las horas de `mascara`."""
    t = ttr[mascara]
    mu_x = Xtr[mascara].mean((0, 1, 2)); sd_x = Xtr[mascara].std((0, 1, 2))
    clim = clim_ponderada(Ytr[mascara], t, cfg['peso_mj'])
    clim_gfs = clim_ponderada(Xtr[mascara], t, cfg['peso_mj'])       # (24,17,13,4)
    anom_sd = (Ytr[mascara] - clim[horas(t)]).std((0, 1, 2))
    D = {'cfg': cfg, 'CLIM': to_t(clim), 'ASD': to_t(anom_sd), 'CLIM_GFS': to_t(clim_gfs),
         'stats': dict(mu_x=mu_x, sd_x=sd_x, clim=clim, anom_sd=anom_sd, clim_gfs=clim_gfs)}
    return D

def preparar_x(D, W, t):
    """Entrada normalizada, hora y término base del residuo (en anomalías normalizadas de WRF)."""
    s = D['stats']
    x = to_t((W - s['mu_x']) / s['sd_x'])
    h = to_t(horas(t)).long()
    modo = D['cfg']['residuo']
    if modo is None:
        return x, h, None
    partes = []
    for i in range(0, len(W), 256):
        gfs = to_t(W[i:i + 256, 2])                                   # hora t, unidades físicas
        hb = h[i:i + 256]
        if modo == 'anomalia':     # pred = clim_WRF + (GFS - clim_GFS) interpolado + red
            b = interpolar((gfs - D['CLIM_GFS'][hb]).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        elif modo == 'crudo':      # pred = GFS interpolado + red
            b = interpolar(gfs.permute(0, 3, 1, 2)).permute(0, 2, 3, 1) - D['CLIM'][hb]
        else:
            raise ValueError(modo)
        partes.append(b / D['ASD'])
    return x, h, torch.cat(partes)

def salida(model, x, h, b):
    p = model(x, h.float())
    return p if b is None else p + b

def red_de(cfg):
    return Net(cfg['skip'], cfg['terreno'], cfg['profundo']).to(dev)

def entrenar(D, tensores, epocas, semilla, lote=32):
    x, h, b, y = tensores
    torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    rng = np.random.default_rng(semilla)
    model = red_de(D['cfg'])
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    n = len(x); pasos = epocas * int(np.ceil(n / lote))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, total_steps=pasos)
    for ep in range(epocas):
        model.train()
        orden = torch.tensor(rng.permutation(n), device=dev)
        for s in range(0, n, lote):
            i = orden[s:s + lote]
            pred = salida(model, x[i], h[i], None if b is None else b[i])
            mse = ((pred - y[i]) ** 2).mean(dim=(0, 1, 2)) * (D['ASD'] / ESC) ** 2
            loss = torch.sqrt(mse + 1e-8).mean()       # pérdida 'rmse' de T08
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model

def tensores_de(D, mascara):
    x, h, b = preparar_x(D, Wtr[mascara], ttr[mascara])
    y = (to_t(Ytr[mascara]) - D['CLIM'][h]) / D['ASD']
    return x, h, b, y

def predecir_lote(model, D, W, t, bs=128):
    x, h, b = preparar_x(D, W, t); model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(W), bs):
            s = slice(i, i + bs)
            p = salida(model, x[s], h[s], None if b is None else b[s])
            out.append((D['CLIM'][h[s]] + p * D['ASD']).cpu().numpy())
    P = np.concatenate(out)
    P[..., 0] = np.clip(P[..., 0], 0, 100)
    return P

def medir(P):
    r = validar(P, Ytr, ttr)
    return {'score_oficial': r['score_oficial'], 'score_local': r['score_local'], 'rmse_3h': r['rmse_horas_evaluadas']}

def linea(etiqueta, m):
    v = ' '.join(f'{k} {x:.3f}' for k, x in m['rmse_3h'].items())
    print(f"{etiqueta:34s} oficial {m['score_oficial']:.5f} · local {m['score_local']:.5f} · {v}", flush=True)

In [ ]:
EPOCAS = [2] if QUICK else [25, 60]
SEMILLAS = [0, 1]
registro = []

def evaluar(nombre, cfg):
    """2 semillas por cada número de épocas; devuelve el mejor número de épocas (por media)."""
    D = datos_de(trn, cfg); T = tensores_de(D, trn)
    por_epocas = {}
    for ep in EPOCAS:
        res = []
        for semilla in SEMILLAS:
            t0 = time.time()
            m = medir(predecir_lote(entrenar(D, T, ep, semilla), D, Wtr[val], ttr[val]))
            linea(f'{nombre} {ep}ep s{semilla} ({time.time() - t0:.0f}s)', m)
            res.append(m)
        por_epocas[ep] = res
    ep = max(por_epocas, key=lambda e: np.mean([m['score_oficial'] for m in por_epocas[e]]))
    s = [m['score_oficial'] for m in por_epocas[ep]]
    r = {'nombre': nombre, 'cfg': cfg, 'epocas': ep, 'semillas': s, 'media': float(np.mean(s)),
         'rmse_3h': {v: float(np.mean([m['rmse_3h'][v] for m in por_epocas[ep]])) for v in VARS},
         'todas': {str(e): v for e, v in por_epocas.items()}}
    del D, T; torch.cuda.empty_cache()
    return r

mejor = evaluar('base', dict(BASE))
mejor['aceptado'] = True; registro.append(mejor)
PASOS = [
    [('skip', dict(skip=True))],
    [('residuo_anomalia', dict(residuo='anomalia')), ('residuo_crudo', dict(residuo='crudo'))],
    [('terreno', dict(terreno=True))],
    [('profundo', dict(profundo=True))],
    [('clim_mj3', dict(peso_mj=3)), ('clim_mj10', dict(peso_mj=10))],
    [('skip_final', dict(skip=True))],          # el skip se probó antes del residuo: se reintenta al final
]
for paso in PASOS:
    anterior = mejor
    candidatos = []
    for nombre, cambio in paso:
        r = evaluar(nombre, {**anterior['cfg'], **cambio})
        r['contra'] = anterior['nombre']
        r['delta'] = r['media'] - anterior['media']
        r['aceptado'] = False
        pasa = r['delta'] >= 0.001 and all(a > b for a, b in zip(r['semillas'], anterior['semillas']))
        candidatos.append((pasa, r)); registro.append(r)
        print(f"== {nombre}: media {r['media']:.5f} ({r['epocas']} ep), delta {r['delta']:+.5f}, pasa={pasa}", flush=True)
    buenos = [r for pasa, r in candidatos if pasa]
    if buenos:
        mejor = max(buenos, key=lambda r: r['media']); mejor['aceptado'] = True
    print('>> configuración vigente:', mejor['nombre'], mejor['cfg'], mejor['epocas'], flush=True)
    json.dump(registro, open('resultados_a.json', 'w'), indent=1)

print(pd.DataFrame([{k: r.get(k) for k in ('nombre', 'contra', 'epocas', 'media', 'delta', 'aceptado')} | r['rmse_3h']
                    for r in registro]).to_string(index=False))

In [ ]:
N_ENS = 2 if QUICK else 5
FINAL = mejor['cfg']; EP = mejor['epocas']
print('Configuración final:', FINAL, EP, 'épocas')

def correr_ensamble(cfg, epocas):
    D = datos_de(trn, cfg); T = tensores_de(D, trn)
    suma, filas, preds = 0, [], None
    for semilla in range(N_ENS):
        P = predecir_lote(entrenar(D, T, epocas, semilla), D, Wtr[val], ttr[val])
        suma = suma + P
        solo, junto = medir(P), medir(suma / (semilla + 1))
        filas.append(dict(semillas=semilla + 1, solo=solo, ensamble=junto))
        linea(f'semilla {semilla} sola', solo); linea(f'ensamble de {semilla + 1}', junto)
    repetida = medir(predecir_lote(entrenar(D, T, epocas, 0), D, Wtr[val], ttr[val]))
    repro = {'primera': filas[0]['solo']['score_oficial'], 'repetida': repetida['score_oficial']}
    del D, T; torch.cuda.empty_cache()
    return suma / N_ENS, filas, repro

print('--- ensamble de la configuración final')
P_nuevo, filas_nuevo, repro = correr_ensamble(FINAL, EP)
print('Reproducibilidad semilla 0:', repro)
print('--- ensamble de la base (referencia T08: 0.45329)')
P_base, filas_base, _ = correr_ensamble(dict(BASE), 25)
mezcla = {str(a): medir(a * P_nuevo + (1 - a) * P_base) for a in (0.5, 0.75)}
for a, m in mezcla.items():
    linea(f'mezcla {a} nuevo + base', m)
json.dump({'final': FINAL, 'epocas': EP, 'ensamble': filas_nuevo, 'base': filas_base, 'mezcla': mezcla,
           'reproducibilidad': repro,
           'versiones': {'python': platform.python_version(), 'numpy': np.__version__, 'torch': torch.__version__,
                         'gpu': torch.cuda.get_device_name(0)}}, open('resultados_b.json', 'w'), indent=1)

In [ ]:
# Reentrenamiento con TODOS los datos (incluye la validación): ya no hay score local que mirar.
todo = np.ones(len(ttr), bool)
D = datos_de(todo, FINAL); T = tensores_de(D, todo)
os.makedirs('models', exist_ok=True)
suma = 0
for semilla in range(N_ENS):
    mod = entrenar(D, T, EP, semilla)
    suma = suma + predecir_lote(mod, D, Wte, tte)
    torch.save(mod.state_dict(), f'models/modelo_s{semilla}.pt')
np.savez('models/estadisticas.npz', **D['stats'])
json.dump({'cfg': FINAL, 'epocas': EP}, open('models/config.json', 'w'))
P = suma / N_ENS
sample = pd.read_csv(find('sample_submission.csv'))
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
sub = pd.DataFrame({'id': sample['id']})
for k, v in enumerate(VARS):
    sub[v] = P[ti, r, c, k].round(3)
assert len(sub) == len(sample) and np.isfinite(sub[list(VARS)].values).all()
sub.to_csv('submission_t09.csv', index=False)
print('submission_t09.csv', sub.shape)